# 1. TTP-enhanced dataset and client-side encoding

This notebook describes the dataset used in RS-FEDRAD and the preprocessing that every client applies locally before federated training. The data are stored per client in `data/clients/`.

In [1]:
import os, sys, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from IPython.display import Image, display
from rsfedrad import config

## 1.1 Clients and class balance

In [2]:
from rsfedrad.data import load_raw_client

rows = []
for i in range(1, 11):
    X, y = load_raw_client(i)
    rows.append({"client": i, "samples": len(y), "ransomware": int(y.sum()), "benign": int((y == 0).sum())})
clients = pd.DataFrame(rows)
clients["ransomware %"] = (clients.ransomware / clients.samples * 100).round(1)
display(clients)
print("total:", clients.samples.sum(), "samples,", clients.ransomware.sum(), "ransomware,", clients.benign.sum(), "benign")

,client,samples,ransomware,benign,ransomware %
0,1,3015,1536,1479,50.9
1,2,3015,1536,1479,50.9
2,3,3015,1536,1479,50.9
3,4,3015,1536,1479,50.9
4,5,3015,1536,1479,50.9
5,6,3015,1536,1479,50.9
6,7,3015,1536,1479,50.9
7,8,3015,1536,1479,50.9
8,9,3015,1536,1479,50.9
9,10,3015,1536,1479,50.9


total: 30150 samples, 15360 ransomware, 14790 benign


## 1.2 Structure of a behavioural sequence

Each sample concatenates 500 API calls, 10 DLLs, 10 mutexes and 10 ATT&CK techniques. Shorter traces leave trailing positions empty.

In [3]:
from rsfedrad.data import load_all_clients

X, y = load_all_clients()
groups = pd.Series([config.group_of(c) for c in X.columns], index=X.columns)
summary = pd.DataFrame({
    "positions": groups.value_counts(sort=False),
    "distinct values": [X.loc[:, groups == g].stack().nunique() for g in groups.unique()],
    "empty (%)": (X.isna().mean().groupby(groups).mean() * 100).round(1),
})
summary

,positions,distinct values,empty (%)
ApiCall,500,272,32.9
Dll,10,1038,31.1
Mutex,10,12576,58.9
TTP,10,15,58.6


In [4]:
i = int(np.argmax(X.filter(like="ApiCall").notna().sum(axis=1).values > 50))
example = X.iloc[i]
print("Label:", config.CLASS_NAMES[y[i]])
print("First API calls:", list(example.filter(like="ApiCall").dropna()[:8]))
print("DLLs:", list(example.filter(like="Dll").dropna()))
print("Mutexes:", list(example.filter(like="Mutex").dropna())[:4])
print("TTPs:", list(example.filter(like="TTP").dropna()))

Label: Ransomware
First API calls: ['PRF', 'PRF', 'PRF', 'SetErrorMode', 'LdrGetDllHandle', 'LdrGetProcedureAddress', 'GetSystemDirectoryW', 'NtAllocateVirtualMemory']
DLLs: ['USER32dll', 'SHELL32dll', 'ole32dll', 'ADVAPI32dll', 'GDI32dll', 'dbghelpdll', 'CRYPT32dll', 'COMCTL32dll', 'VERSIONdll', 'NETAPI32dll']
Mutexes: ['LocalZonesCacheCounterMutex', 'LocalZoneAttributeCacheCounterMutex', 'LocalZonesLockedCacheCounterMutex']
TTPs: ['Unknown']


## 1.3 ATT&CK techniques by class

In [5]:
from rsfedrad.explain import ATTACK_TECHNIQUES

ttp = X.filter(like="TTP").assign(label=y).melt(id_vars="label", value_name="technique").dropna()
counts = (ttp.groupby(["technique", "label"]).size().unstack(fill_value=0)
          .rename(columns={0: "benign", 1: "ransomware"}))
counts["name"] = [ATTACK_TECHNIQUES.get(t, "") for t in counts.index]
counts.sort_values("ransomware", ascending=False)

label,benign,ransomware,name
technique,,,
T1055,0,9322,Process Injection
T1106,4269,9082,Native API
T1012,0,8994,Query Registry
T1027,0,8646,Obfuscated Files or Information
T1098,0,8635,Account Manipulation
T1134,0,8450,Access Token Manipulation
T1003,0,8412,OS Credential Dumping
T1021,0,8411,Remote Services
T1073,0,8411,DLL Side-Loading


## 1.4 Client-side encoding

Every client imputes empty positions with the most frequent value, label-encodes each position, scales the codes to [0, 1] and standardises them. The encoder retains the vocabulary of every position, so encoded inputs can be mapped back to readable behaviour, which the explainability analysis relies on.

In [6]:
from rsfedrad.data import load_client

c = load_client(1)
print("train:", c.X_train.shape, " test:", c.X_test.shape)
print("ransomware share train/test: %.3f / %.3f" % (c.y_train.mean(), c.y_test.mean()))
row = c.idx_test[0]
decoded = c.encoder.decode_row(c.codes[row])
print({k: decoded[k] for k in ["ApiCall_0", "ApiCall_1", "Dll_0", "Mutex_0", "TTP_0"]})

train: (2412, 530, 1)  test: (603, 530, 1)
ransomware share train/test: 0.510 / 0.509
{'ApiCall_0': 'GetSystemTimeAsFileTime', 'ApiCall_1': 'SetUnhandledExceptionFilter', 'Dll_0': 'USER32dll', 'Mutex_0': 'GlobalMCStoreSyncMem71bdfe29063ac557a4e7b3205ed180408457fcd4', 'TTP_0': 'Unknown'}
